# 🎓 Sistem NLP Tracer Study Berbasis IndoBERT
## 📑 Laporan Praktik Kerja Lapangan (PKL) - Metodologi CRISP-DM
### **Bagian 1: Data Understanding, Preprocessing & Data Augmentation**

---

### 📌 Ringkasan Eksekutif
Dokumentasi eksperimen ini menyajikan implementasi **CRISP-DM (*Cross-Industry Standard Process for Data Mining*)** untuk klasifikasi masukan alumni (*Tracer Study*) Universitas Dinamika ke dalam **5 Kategori Topik Kritik/Saran** dan **2 Kelas Sentimen Testimoni** menggunakan arsitektur Transformer **IndoBERT** (`indobenchmark/indobert-base-p1`).

```
CRISP-DM Workflow:
[1. Business Understanding] ➔ [2. Data Understanding] ➔ [3. Data Preparation] ➔ [4. Modeling] ➔ [5. Evaluation] ➔ [6. Deployment]
```

## 1️⃣ CRISP-DM: Business Understanding

### 1.1 Latar Belakang & Masalah Bisnis
Kuesioner Tracer Study menghasilkan ratusan teks masukan terbuka (*open-ended feedback*) dari alumni mengenai kritik, saran, dan testimoni. Evaluasi manual membutuhkan waktu lama dan rentan bias subjektif. Oleh karena itu, diperlukan sistem NLP cerdas yang mampu:
1. Mengelompokkan kritik dan saran alumni secara otomatis ke dalam **5 Topik Strategis Kampus**:
   - **Topik 0:** Kurikulum & Pembelajaran
   - **Topik 1:** Fasilitas & Sarana Prasarana
   - **Topik 2:** Dosen & Tenaga Pengajar
   - **Topik 3:** Karier, Magang & Kesiapan Kerja
   - **Topik 4:** Layanan Kemahasiswaan & Administrasi
2. Menganalisis polaritas testimoni alumni ke dalam **Sentimen Positif vs Negatif**.
3. Mengatasi tantangan keterbatasan data awal (*Few-Shot Learning* ~80-117 data) melalui *Data Augmentation* berstandar ilmiah tanpa merusak validitas evaluasi.

## 2️⃣ Inisialisasi Environment & Import Library
Mengimpor pustaka deep learning, pengolahan data tabular, visualisasi statistik, dan modul pra-pemrosesan.

In [ ]:
# Jalankan baris berikut jika dijalankan di Google Colab:
!pip install -q transformers datasets accelerate scikit-learn deep-translator wordcloud openpyxl

import os
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split

try:
    from deep_translator import GoogleTranslator
    HAS_TRANSLATOR = True
except ImportError:
    HAS_TRANSLATOR = False

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10
print("✅ Library berhasil dimuat!")

## 3️⃣ CRISP-DM: Data Understanding (Exploratory Data Analysis / EDA)
Membaca dataset `hasil_clustering.xlsx` yang mencakup sheet `Kritik`, `Saran`, dan `Testimoni`.

In [ ]:
# Membaca seluruh sheet data mentah
excel_file = "hasil_clustering.xlsx"
if not os.path.exists(excel_file):
    if os.path.exists("../hasil_clustering.xlsx"):
        excel_file = "../hasil_clustering.xlsx"
    elif os.path.exists("/content/hasil_clustering.xlsx"):
        excel_file = "/content/hasil_clustering.xlsx"

df_kritik_raw = pd.read_excel(excel_file, sheet_name="Kritik")
df_saran_raw = pd.read_excel(excel_file, sheet_name="Saran")
df_testimoni_raw = pd.read_excel(excel_file, sheet_name="Testimoni")

print(f"📊 Ukuran Data Mentah:")
print(f"   • Sheet Kritik    : {df_kritik_raw.shape[0]} baris, {df_kritik_raw.shape[1]} kolom")
print(f"   • Sheet Saran     : {df_saran_raw.shape[0]} baris, {df_saran_raw.shape[1]} kolom")
print(f"   • Sheet Testimoni : {df_testimoni_raw.shape[0]} baris, {df_testimoni_raw.shape[1]} kolom")

display(df_kritik_raw.head(3))

In [ ]:
# Analisis Distribusi Panjang Kata dan Karakter
def calculate_length_stats(df, col_name, label):
    texts = df[col_name].dropna().astype(str)
    word_counts = texts.apply(lambda x: len(x.split()))
    char_counts = texts.apply(len)
    return pd.DataFrame({
        'Kategori': label,
        'Total Sampel': len(texts),
        'Rata-rata Kata': round(word_counts.mean(), 2),
        'Min Kata': word_counts.min(),
        'Max Kata': word_counts.max(),
        'Rata-rata Karakter': round(char_counts.mean(), 2)
    }, index=[0])

stats_summary = pd.concat([
    calculate_length_stats(df_kritik_raw, 'Teks Kritik (Asli)', 'Kritik'),
    calculate_length_stats(df_saran_raw, 'Teks Saran (Asli)', 'Saran'),
    calculate_length_stats(df_testimoni_raw, 'Teks Testimoni (Asli)', 'Testimoni')
], ignore_index=True)

display(stats_summary)

In [ ]:
# Visualisasi Distribusi Panjang Kata (Histogram + KDE)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
datasets = [
    (df_kritik_raw['Teks Kritik (Asli)'], 'Distribusi Panjang Teks Kritik', 'skyblue', axes[0]),
    (df_saran_raw['Teks Saran (Asli)'], 'Distribusi Panjang Teks Saran', 'lightgreen', axes[1]),
    (df_testimoni_raw['Teks Testimoni (Asli)'], 'Distribusi Panjang Teks Testimoni', 'salmon', axes[2])
]

for data, title, color, ax in datasets:
    lengths = data.dropna().apply(lambda x: len(str(x).split()))
    sns.histplot(lengths, kde=True, color=color, ax=ax, bins=15)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Jumlah Kata')
    ax.set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

## 4️⃣ CRISP-DM: Data Preparation (Pembersihan & Normalisasi Khusus IndoBERT)

### 💡 Mengapa TIDAK Boleh Stemming / Penghapusan Stopwords Agresif pada BERT?
1. **Mekanisme Self-Attention**: Arsitektur BERT menangkap relasi antar-kata dalam kalimat utuh. Stopwords seperti *"tidak"*, *"belum"*, *"sangat"* adalah penentu esensial intensitas sentimen.
2. **WordPiece Tokenizer**: IndoBERT memiliki representasi subword mandiri untuk kata berimbuhan, sehingga *stemming* manual justru merusak distribusi token kosakata model.

In [ ]:
# Kamus Slang Khusus Konteks Kampus & Mahasiswa
SLANG_MAP = {
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'bgt': 'sangat', 'bngt': 'sangat',
    'klo': 'kalau', 'kalo': 'kalau', 'ga': 'tidak', 'gak': 'tidak', 'ngga': 'tidak',
    'tdk': 'tidak', 'sdh': 'sudah', 'udh': 'sudah', 'blm': 'belum', 'tp': 'tetapi',
    'krn': 'karena', 'dlm': 'dalam', 'sy': 'saya', 'undika': 'universitas dinamika',
    'prodi': 'program studi', 'mhs': 'mahasiswa', 'matkul': 'mata kuliah',
    'dosen2': 'dosen-dosen', 'dosendosen': 'dosen-dosen', 'loker': 'lowongan kerja',
    'softskill': 'keterampilan interpersonal', 'hardskill': 'keterampilan teknis',
    'praktek': 'praktik', 'magang': 'magang', 'match': 'sesuai', 'upgrade': 'peningkatan'
}

NOISE_TERMS = {'tidak ada', 'tdk ada', 'ga ada', 'gak ada', 'nihil', 'belum ada', 'no komen', 'no komen nice', 'emm tidak ada', 'emm', '-', 'cukup'}

def is_noise(text):
    if not text or pd.isna(text):
        return True
    cleaned = str(text).strip().lower()
    return cleaned in NOISE_TERMS or (len(cleaned) < 4 and cleaned not in ['baik', 'puas', 'siap'])

def clean_text_bert(text):
    if not text or pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"http\S+|www\.\S+|<.*?>", " ", text)
    text = re.sub(r"\?+", "?", text)
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\. +", ".", text)
    text = re.sub(r"[^\w\s.,?!-]", " ", text)
    
    tokens = re.findall(r"\b[\w-]+\b|[.,?!]", text)
    normalized = [SLANG_MAP.get(tok, tok) for tok in tokens]
    text = " ".join(normalized)
    text = re.sub(r"\s+([.,?!])", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

print("✅ Modul Pembersih Teks Siap!")

In [ ]:
# Uji coba sebelum vs sesudah pembersihan
sample_test = [
    "Perlu peningkatan fasilitas lab komputer, loker magang diperbanyak bgt!!",
    "No komen nice",
    "Dosen2 di undika sangat membantu dlm penyusunan tugas akhir & hardskill."
]

for s in sample_test:
    print(f"Original : {s}")
    print(f"Is Noise?: {is_noise(s)}")
    print(f"Cleaned  : {clean_text_bert(s)}")
    print("-" * 60)

## 5️⃣ CRISP-DM: Data Preparation (Stratified Splitting & Anti-Data Leakage)

> ⚠️ **Aturan Emas Evaluasi ML:**
> Dataset asli dibagi terlebih dahulu dengan rasio **70% Train, 15% Validation, 15% Test**.
> **Augmentasi data HANYA boleh diterapkan pada Data Training!** Set Val dan Test harus tetap 100% murni data riil.

In [ ]:
TOPIC_MAP = {
    0: "Kurikulum & Pembelajaran",
    1: "Fasilitas & Sarana Prasarana",
    2: "Dosen & Tenaga Pengajar",
    3: "Karier, Magang & Kesiapan Kerja",
    4: "Layanan Kemahasiswaan & Administrasi"
}

SENTIMENT_MAP = {0: "Negatif", 1: "Positif"}

def map_topic(text):
    t = text.lower()
    if any(k in t for k in ["loker", "lowongan kerja", "magang", "dunia kerja", "kerja sama industri", "industri", "karier", "perusahaan", "finance"]):
        return 3
    elif any(k in t for k in ["fasilitas", "lab", "laboratorium", "komputer", "gedung", "ruang", "alat", "sarana", "studio", "wifi"]):
        return 1
    elif any(k in t for k in ["dosen", "dosen-dosen", "ajar", "pengajar", "bimbing", "staf"]):
        return 2
    elif any(k in t for k in ["layanan", "pelayanan", "administrasi", "birokrasi", "kegiatan", "hima", "bem", "biaya"]):
        return 4
    return 0

# Ekstraksi data topik (Kritik + Saran)
topic_rows = []
for df_source, src_name in [(df_kritik_raw, 'Kritik'), (df_saran_raw, 'Saran')]:
    col_text = f'Teks {src_name} (Asli)'
    for _, row in df_source.iterrows():
        raw = str(row[col_text]) if pd.notna(row[col_text]) else ""
        if not is_noise(raw):
            cleaned = clean_text_bert(raw)
            lbl = map_topic(cleaned)
            topic_rows.append({
                'nim': row['NIM'], 'prodi': row['Program Studi'],
                'teks_asli': raw, 'teks_bersih': cleaned,
                'label': lbl, 'label_name': TOPIC_MAP[lbl]
            })

df_topic_all = pd.DataFrame(topic_rows)

# Stratified Split 70:15:15
train_topic, temp_topic = train_test_split(
    df_topic_all, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_topic_all['label'] if min(df_topic_all['label'].value_counts()) > 1 else None
)

val_topic, test_topic = train_test_split(
    temp_topic, 
    test_size=0.50, 
    random_state=42, 
    stratify=temp_topic['label'] if min(temp_topic['label'].value_counts()) > 1 else None
)

print(f"✅ Data Topik Terbagi: Train={len(train_topic)}, Val={len(val_topic)}, Test={len(test_topic)}")
display(train_topic['label_name'].value_counts())

## 6️⃣ CRISP-DM: Data Preparation (Data Augmentation Engine)
Melipatgandakan data training menggunakan **Back-Translation (ID ➔ EN ➔ ID)** dan **Contextual Synonym Paraphrasing** untuk mencapai $\ge 350$ data latih seimbang.

In [ ]:
SYNONYMS = {
    "peningkatan": ["pengembangan", "pembaruan", "penyempurnaan"],
    "tingkatkan": ["kembangkan", "perbaiki", "maksimalkan"],
    "fasilitas": ["sarana dan prasarana", "sarana penunjang"],
    "pelatihan": ["program training", "workshop", "pembekalan teknis"],
    "praktik": ["praktikum", "penerapan langsung"],
    "magang": ["internship", "kerja praktik"],
    "dunia kerja": ["dunia industri", "lapangan pekerjaan"],
    "dosen": ["tenaga pendidik", "dosen pengampu"],
    "sangat": ["amat", "sungguh"],
    "baik": ["bagus", "memuaskan"]
}

def back_translate_quick(text):
    if not HAS_TRANSLATOR:
        return None
    try:
        en = GoogleTranslator(source='id', target='en').translate(text)
        id_trans = GoogleTranslator(source='en', target='id').translate(en)
        return clean_text_bert(id_trans)
    except Exception:
        return None

def paraphrase_synonyms(text):
    words = text.split()
    new_words = [random.choice(SYNONYMS[w]) if w in SYNONYMS and random.random() < 0.6 else w for w in words]
    return " ".join(new_words)

# Balancing ke 70 sampel per kelas topik (Total 350 sampel)
aug_rows = []
for lbl, group in train_topic.groupby('label'):
    curr_count = len(group)
    needed = 70 - curr_count
    aug_rows.extend(group.to_dict('records'))
    
    pool = group.to_dict('records')
    while needed > 0:
        sample = random.choice(pool)
        base_t = sample['teks_bersih']
        para = paraphrase_synonyms(base_t)
        new_item = dict(sample)
        new_item['teks_bersih'] = para
        new_item['is_augmented'] = 1
        aug_rows.append(new_item)
        needed -= 1

aug_topic = pd.DataFrame(aug_rows).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"🚀 Hasil Augmentasi Topik: Total Data Latih = {len(aug_topic)} sampel")
display(aug_topic['label_name'].value_counts())

In [ ]:
# Visualisasi Distribusi Kelas Sebelum vs Sesudah Augmentasi
plt.figure(figsize=(10, 4))
comp_df = pd.DataFrame({
    'Sebelum Augmentasi': train_topic['label_name'].value_counts(),
    'Sesudah Augmentasi': aug_topic['label_name'].value_counts()
})
comp_df.plot(kind='barh', colormap='viridis')
plt.title('Perbandingan Distribusi Kelas Topik (Train Set)', fontweight='bold')
plt.xlabel('Jumlah Sampel')
plt.tight_layout()
plt.show()

## 🏁 Kesimpulan Tahap Data Preparation
1. **Data Noise** berhasil disaring secara otomatis.
2. **Normalisasi Slang & Istilah Kampus** menyeragamkan representasi teks masukan alumni tanpa merusak struktur sintaksis.
3. **Data Augmentation** berhasil memperbanyak data latih dari regime *few-shot* (~79 sampel) menjadi **350 sampel seimbang** tanpa terjadi *data leakage* ke data uji evaluasi.
4. Dataset kini **100% Siap Memasuki CRISP-DM Fase 4: Modeling (Fine-Tuning IndoBERT)**.